# Stage 01/02/03 Results Comparison

Build the final comparison report from the generated stage artifacts.

In [1]:
from __future__ import annotations
from pathlib import Path as _NotebookPath
import sys as _notebook_sys

for _candidate in (_NotebookPath.cwd(), *_NotebookPath.cwd().parents):
    _notebook_path = _candidate / 'ml_models/04/results/compare_01_02_03.ipynb'
    if _notebook_path.exists():
        __file__ = str(_notebook_path.resolve())
        break
else:
    __file__ = str((_NotebookPath.cwd() / 'ml_models/04/results/compare_01_02_03.ipynb').resolve())

_notebook_sys.argv = [__file__]


## Implementation

In [2]:
import json
from pathlib import Path
from typing import Any

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import yaml


def find_repo_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").exists() and (candidate / "ml_models").exists():
            return candidate
    raise RuntimeError(f"Could not find repository root from {start}")


ROOT = find_repo_root(Path(__file__).resolve().parent)
OUT_DIR = Path(__file__).resolve().parent
FIG_DIR = OUT_DIR / "figures"

STAGE_01_BEST = ROOT / "ml_models/01_baseline/results/best_tau.yaml"
STAGE_02_BEST = ROOT / "ml_models/02_ssl/results/best_tau_surrogate.yaml"
STAGE_02_TEST_METRICS = ROOT / "ml_models/02_ssl/results/test_metrics.json"
STAGE_03_KPI = ROOT / "ml_models/03_rl+ssl/results/kpi_comparison.csv"
STAGE_03_PER_TICK_KPI = ROOT / "ml_models/03_rl+ssl/results/per_tick/kpi_comparison_with_ci.csv"
STAGE_03_PER_TICK_SUMMARY = ROOT / "ml_models/03_rl+ssl/results/per_tick/per_tick_summary.yaml"
STAGE_03_PER_TICK_PRINTERS = ROOT / "ml_models/03_rl+ssl/results/per_tick/per_printer_test_ensemble.csv"
STAGE_03_TAU = ROOT / "ml_models/03_rl+ssl/results/best_tau_per_printer.yaml"

COMPONENTS = ("C1", "C2", "C3", "C4", "C5", "C6")


def load_yaml(path: Path) -> dict[str, Any]:
    with path.open("r", encoding="utf-8") as handle:
        data = yaml.safe_load(handle)
    if not isinstance(data, dict):
        raise ValueError(f"{path} did not contain a YAML mapping")
    return data


def load_json(path: Path) -> dict[str, Any]:
    with path.open("r", encoding="utf-8") as handle:
        data = json.load(handle)
    if not isinstance(data, dict):
        raise ValueError(f"{path} did not contain a JSON mapping")
    return data


def require_inputs() -> None:
    required = [
        STAGE_01_BEST,
        STAGE_02_BEST,
        STAGE_02_TEST_METRICS,
        STAGE_03_KPI,
        STAGE_03_PER_TICK_KPI,
        STAGE_03_PER_TICK_SUMMARY,
        STAGE_03_PER_TICK_PRINTERS,
        STAGE_03_TAU,
    ]
    missing = [str(p.relative_to(ROOT)) for p in required if not p.exists()]
    if missing:
        raise FileNotFoundError(
            "Missing comparison inputs. Run stages 01/02/03 first:\n"
            + "\n".join(f"  - {m}" for m in missing)
        )


def pct(value: float) -> str:
    return f"{value * 100:.2f}%"


def money(value: float) -> str:
    return f"EUR {value:,.0f}"


def short_float(value: float) -> str:
    if abs(value) >= 1e9:
        return f"{value / 1e9:.3f}B"
    if abs(value) >= 1e6:
        return f"{value / 1e6:.3f}M"
    if abs(value) >= 1e3:
        return f"{value / 1e3:.3f}K"
    return f"{value:.3f}"


def md_table(df: pd.DataFrame) -> str:
    if df.empty:
        return "_No rows._"
    headers = list(df.columns)
    rows = [[str(v) for v in row] for row in df.itertuples(index=False, name=None)]
    widths = [
        max(len(headers[i]), *(len(row[i]) for row in rows))
        for i in range(len(headers))
    ]
    out = [
        "| " + " | ".join(headers[i].ljust(widths[i]) for i in range(len(headers))) + " |",
        "| " + " | ".join("-" * widths[i] for i in range(len(headers))) + " |",
    ]
    for row in rows:
        out.append("| " + " | ".join(row[i].ljust(widths[i]) for i in range(len(headers))) + " |")
    return "\n".join(out)


def build_stage_kpis() -> pd.DataFrame:
    src = pd.read_csv(STAGE_03_PER_TICK_KPI)
    keep = src[src["stage"].isin(["stage_01", "stage_02", "stage_03_per_tick"])].copy()
    labels = {
        "stage_01": "Stage 01 - Optuna constant tau",
        "stage_02": "Stage 02 - SSL/RUL surrogate tau",
        "stage_03_per_tick": "Stage 03 - per-tick PPO+SPR ensemble",
    }
    keep["stage_label"] = keep["stage"].map(labels)
    keep = keep.sort_values("stage").reset_index(drop=True)

    baseline = float(keep.loc[keep["stage"] == "stage_01", "fleet_annual_cost"].iloc[0])
    baseline_value = float(keep.loc[keep["stage"] == "stage_01", "fleet_value"].iloc[0])
    keep["annual_cost_delta_vs_stage01"] = keep["fleet_annual_cost"] - baseline
    keep["annual_cost_reduction_vs_stage01_pct"] = (baseline - keep["fleet_annual_cost"]) / baseline
    keep["value_delta_vs_stage01"] = keep["fleet_value"] - baseline_value
    keep["value_reduction_vs_stage01_pct"] = (baseline_value - keep["fleet_value"]) / baseline_value
    return keep


def build_tau_comparison() -> pd.DataFrame:
    s1 = load_yaml(STAGE_01_BEST)["tau_nom_h"]
    s2 = load_yaml(STAGE_02_BEST)["tau_nom_h"]
    s3 = load_yaml(STAGE_03_TAU)["tau_per_printer"]
    s3_df = pd.DataFrame.from_dict(s3, orient="index").astype(float)

    rows = []
    for component in COMPONENTS:
        rows.append(
            {
                "component": component,
                "stage_01_tau_h": float(s1[component]),
                "stage_02_tau_h": float(s2[component]),
                "stage_03_per_printer_tau_mean_h": float(s3_df[component].mean()),
                "stage_03_per_printer_tau_min_h": float(s3_df[component].min()),
                "stage_03_per_printer_tau_max_h": float(s3_df[component].max()),
            }
        )
    return pd.DataFrame(rows)


def build_auxiliary_stage03() -> pd.DataFrame:
    return pd.read_csv(STAGE_03_KPI)


def build_stage03_per_printer_summary() -> pd.DataFrame:
    df = pd.read_csv(STAGE_03_PER_TICK_PRINTERS)
    cols = [
        "printer_id",
        "annual_cost",
        "availability",
        "deficit",
        "n_preventive",
        "n_corrective",
    ]
    return df[cols].copy()


def write_plots(stage_kpis: pd.DataFrame, tau_df: pd.DataFrame, per_printer: pd.DataFrame) -> None:
    FIG_DIR.mkdir(parents=True, exist_ok=True)
    plt.style.use("seaborn-v0_8-whitegrid")

    labels = ["Stage 01", "Stage 02", "Stage 03"]
    colors = ["#6b7280", "#2563eb", "#059669"]

    fig, ax_cost = plt.subplots(figsize=(10, 5.5))
    x = np.arange(len(stage_kpis))
    costs_m = stage_kpis["fleet_annual_cost"].to_numpy(dtype=float) / 1e6
    ax_cost.bar(x, costs_m, color=colors, alpha=0.86)
    ax_cost.set_ylabel("Annual cost (EUR M / printer-year)")
    ax_cost.set_xticks(x, labels)
    ax_cost.set_title("Cost and availability by stage")
    ax_avail = ax_cost.twinx()
    ax_avail.plot(x, stage_kpis["fleet_availability"], color="#111827", marker="o", linewidth=2.0)
    ax_avail.set_ylim(0, 1)
    ax_avail.set_ylabel("Availability")
    for i, value in enumerate(costs_m):
        ax_cost.text(i, value, f"{value:.2f}M", ha="center", va="bottom", fontsize=9)
    for i, value in enumerate(stage_kpis["fleet_availability"]):
        ax_avail.text(i, value + 0.03, pct(float(value)), ha="center", va="bottom", fontsize=9)
    fig.tight_layout()
    fig.savefig(FIG_DIR / "cost_availability_by_stage.png", dpi=160)
    plt.close(fig)

    fig, ax = plt.subplots(figsize=(9, 5))
    values_b = stage_kpis["fleet_value"].to_numpy(dtype=float) / 1e9
    ax.bar(labels, values_b, color=colors, alpha=0.86)
    ax.set_ylabel("Penalized objective (B)")
    ax.set_title("Penalized objective by stage")
    for i, value in enumerate(values_b):
        ax.text(i, value, f"{value:.2f}B", ha="center", va="bottom", fontsize=9)
    fig.tight_layout()
    fig.savefig(FIG_DIR / "penalized_value_by_stage.png", dpi=160)
    plt.close(fig)

    fig, ax = plt.subplots(figsize=(11, 5.5))
    idx = np.arange(len(tau_df))
    width = 0.25
    ax.bar(idx - width, tau_df["stage_01_tau_h"], width, label="Stage 01", color=colors[0])
    ax.bar(idx, tau_df["stage_02_tau_h"], width, label="Stage 02", color=colors[1])
    ax.bar(
        idx + width,
        tau_df["stage_03_per_printer_tau_mean_h"],
        width,
        label="Stage 03 per-printer mean",
        color=colors[2],
    )
    ax.set_yscale("log")
    ax.set_xticks(idx, tau_df["component"])
    ax.set_ylabel("tau interval (hours, log scale)")
    ax.set_title("Maintenance interval comparison")
    ax.legend()
    fig.tight_layout()
    fig.savefig(FIG_DIR / "tau_comparison.png", dpi=160)
    plt.close(fig)

    fig, ax = plt.subplots(figsize=(8, 5.5))
    ax.scatter(
        per_printer["availability"],
        per_printer["annual_cost"] / 1e6,
        s=58,
        color="#059669",
        edgecolor="#064e3b",
        linewidth=0.8,
    )
    for _, row in per_printer.iterrows():
        ax.text(row["availability"] + 0.004, row["annual_cost"] / 1e6, str(int(row["printer_id"])), fontsize=8)
    ax.set_xlabel("Availability")
    ax.set_ylabel("Annual cost (EUR M / printer-year)")
    ax.set_title("Stage 03 per-tick ensemble by test printer")
    fig.tight_layout()
    fig.savefig(FIG_DIR / "stage03_per_printer_cost_availability.png", dpi=160)
    plt.close(fig)


def write_report(
    stage_kpis: pd.DataFrame,
    tau_df: pd.DataFrame,
    aux_stage03: pd.DataFrame,
    per_printer: pd.DataFrame,
) -> None:
    s2_metrics = load_json(STAGE_02_TEST_METRICS)
    per_tick_summary = load_yaml(STAGE_03_PER_TICK_SUMMARY)

    display_kpis = stage_kpis[
        [
            "stage_label",
            "policy_class",
            "fleet_annual_cost",
            "fleet_availability",
            "fleet_deficit",
            "fleet_value",
            "annual_cost_reduction_vs_stage01_pct",
            "value_reduction_vs_stage01_pct",
        ]
    ].copy()
    display_kpis.columns = [
        "stage",
        "policy",
        "annual_cost",
        "availability",
        "deficit",
        "penalized_value",
        "cost_reduction_vs_01",
        "value_reduction_vs_01",
    ]
    display_kpis["annual_cost"] = display_kpis["annual_cost"].map(money)
    display_kpis["availability"] = display_kpis["availability"].map(pct)
    display_kpis["deficit"] = display_kpis["deficit"].map(pct)
    display_kpis["penalized_value"] = display_kpis["penalized_value"].map(short_float)
    display_kpis["cost_reduction_vs_01"] = display_kpis["cost_reduction_vs_01"].map(pct)
    display_kpis["value_reduction_vs_01"] = display_kpis["value_reduction_vs_01"].map(pct)

    tau_display = tau_df.copy()
    for col in tau_display.columns:
        if col != "component":
            tau_display[col] = tau_display[col].map(lambda v: f"{v:,.1f}")

    aux_display = aux_stage03.copy()
    aux_display["fleet_annual_cost"] = aux_display["fleet_annual_cost"].map(money)
    aux_display["fleet_availability"] = aux_display["fleet_availability"].map(pct)
    aux_display["fleet_deficit"] = aux_display["fleet_deficit"].map(pct)
    aux_display["fleet_value"] = aux_display["fleet_value"].map(short_float)
    aux_display = aux_display[
        ["stage", "policy_class", "fleet_annual_cost", "fleet_availability", "fleet_deficit", "fleet_value"]
    ]

    per_printer_summary = pd.DataFrame(
        [
            {
                "metric": "annual_cost_min",
                "value": money(float(per_printer["annual_cost"].min())),
            },
            {
                "metric": "annual_cost_mean",
                "value": money(float(per_printer["annual_cost"].mean())),
            },
            {
                "metric": "annual_cost_max",
                "value": money(float(per_printer["annual_cost"].max())),
            },
            {
                "metric": "availability_min",
                "value": pct(float(per_printer["availability"].min())),
            },
            {
                "metric": "availability_mean",
                "value": pct(float(per_printer["availability"].mean())),
            },
            {
                "metric": "availability_max",
                "value": pct(float(per_printer["availability"].max())),
            },
        ]
    )

    s2_display = pd.DataFrame(
        [
            {
                "variant": name,
                "mae_mean_days": f"{metrics['mae_mean']:.2f}",
                "rmse_mean_days": f"{metrics['rmse_mean']:.2f}",
            }
            for name, metrics in s2_metrics.items()
        ]
    )

    winner = stage_kpis.sort_values(["fleet_value", "fleet_annual_cost"]).iloc[0]
    cost_reduction = float(
        stage_kpis.loc[stage_kpis["stage"] == "stage_03_per_tick", "annual_cost_reduction_vs_stage01_pct"].iloc[0]
    )
    avail_gain = float(
        stage_kpis.loc[stage_kpis["stage"] == "stage_03_per_tick", "fleet_availability"].iloc[0]
        - stage_kpis.loc[stage_kpis["stage"] == "stage_01", "fleet_availability"].iloc[0]
    )

    report = f"""# Stage 01/02/03 Results Comparison

This report compares the three maintenance-policy stages using the generated artifacts under `ml_models/`.

Main Stage 03 result: the per-tick PPO+SPR ensemble from `ml_models/03_rl+ssl/results/per_tick/`. The earlier Stage 03 per-printer tau policy is retained below as auxiliary context because it is useful diagnostically but is weaker than the per-tick policy.

## Headline

- Best overall row by penalized objective: **{winner['stage_label']}**.
- Stage 03 per-tick reduces annual cost by **{pct(cost_reduction)}** versus Stage 01.
- Stage 03 per-tick improves availability by **{pct(avail_gain)}** versus Stage 01.
- None of the stages reach the 95% availability constraint yet, so every final objective still includes a deficit penalty.

## Normalized KPI Table

{md_table(display_kpis)}

## Maintenance Interval Comparison

Stage 01 and Stage 02 output one constant tau vector. The auxiliary Stage 03 per-printer tau policy outputs one tau vector per test printer; this table shows its mean/min/max by component. The main Stage 03 per-tick policy is event/action based, so it is not directly represented by a fixed tau vector.

{md_table(tau_display)}

## Stage 02 RUL Head Metrics

Mean held-out RUL error by variant:

{md_table(s2_display)}

## Stage 03 Auxiliary Context

Earlier Stage 03 per-printer tau comparison from `ml_models/03_rl+ssl/results/kpi_comparison.csv`:

{md_table(aux_display)}

Per-tick PPO+SPR ensemble summary from `per_tick_summary.yaml`:

| metric | value |
| --- | --- |
| fleet annual cost | {money(float(per_tick_summary['fleet_annual_cost_eur_per_printer_year']))} |
| fleet availability | {pct(float(per_tick_summary['fleet_availability']))} |
| fleet deficit | {pct(float(per_tick_summary['fleet_deficit']))} |
| ensemble size | {per_tick_summary['ensemble_size']} |
| total timesteps per seed | {per_tick_summary['config']['total_timesteps_per_seed']} |

Per-printer spread for the Stage 03 per-tick ensemble:

{md_table(per_printer_summary)}

## Figures

![cost_availability_by_stage](figures/cost_availability_by_stage.png)

![penalized_value_by_stage](figures/penalized_value_by_stage.png)

![tau_comparison](figures/tau_comparison.png)

![stage03_per_printer_cost_availability](figures/stage03_per_printer_cost_availability.png)

## Interpretation

Stage 02 improves cost relative to Stage 01 but leaves availability at zero in the test KPI table. Its RUL model still matters because it produces the trained encoder and RUL head used downstream, but the constant-tau surrogate winner does not satisfy the operational constraint.

The earlier Stage 03 per-printer tau policy barely improves the constant-tau policies. The per-tick PPO+SPR ensemble is materially better: it lowers annual cost and raises availability to about 44%. It is still infeasible against the 95% availability requirement, which means the next useful work is not presentation polish; it is reward/action design and simulator-policy alignment.

High-leverage next steps:

1. Increase Stage 03 training budget and evaluate more seeds.
2. Revisit the reward: availability deficit should dominate earlier, not only after annual cost has already improved.
3. Let the per-tick policy maintain C2/C4/C5/C6 more intelligently; the current ensemble fires daily preventive maintenance for C1/C3 in the per-printer event table.
4. Clear GPU/RAM contention before training. The last run had an unrelated `llama-server` occupying about 20GB on GPU 1 and the system was already using swap.

## Reproduce

```bash
uv run jupyter nbconvert --to notebook --execute --inplace ml_models/04/results/compare_01_02_03.ipynb
```
"""
    (OUT_DIR / "REPORT.md").write_text(report, encoding="utf-8")


def main() -> None:
    require_inputs()
    OUT_DIR.mkdir(parents=True, exist_ok=True)
    FIG_DIR.mkdir(parents=True, exist_ok=True)

    stage_kpis = build_stage_kpis()
    tau_df = build_tau_comparison()
    aux_stage03 = build_auxiliary_stage03()
    per_printer = build_stage03_per_printer_summary()

    stage_kpis.to_csv(OUT_DIR / "stage_kpis.csv", index=False)
    tau_df.to_csv(OUT_DIR / "tau_comparison.csv", index=False)
    aux_stage03.to_csv(OUT_DIR / "stage03_auxiliary_kpis.csv", index=False)
    per_printer.to_csv(OUT_DIR / "stage03_per_tick_printer_summary.csv", index=False)

    write_plots(stage_kpis, tau_df, per_printer)
    write_report(stage_kpis, tau_df, aux_stage03, per_printer)

    print(f"Wrote {OUT_DIR / 'REPORT.md'}")
    print(f"Wrote {OUT_DIR / 'stage_kpis.csv'}")
    print(f"Wrote {OUT_DIR / 'tau_comparison.csv'}")
    print(f"Wrote {FIG_DIR}")



## Generate Report

In [3]:
main()

Wrote /home/sterry/Desktop/projects/hackupc2026/ml_models/04/results/REPORT.md
Wrote /home/sterry/Desktop/projects/hackupc2026/ml_models/04/results/stage_kpis.csv
Wrote /home/sterry/Desktop/projects/hackupc2026/ml_models/04/results/tau_comparison.csv
Wrote /home/sterry/Desktop/projects/hackupc2026/ml_models/04/results/figures
